In [1]:
import numpy as np
from dataclasses import dataclass
from scipy.optimize import minimize
from scipy.stats import qmc

# ============================================================
# Helper functions
# ============================================================

def project_to_bounds(x: np.ndarray, bounds: np.ndarray) -> np.ndarray:
    """Project x into [low, high] box constraints."""
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    return np.minimum(np.maximum(x, lows), highs)

def make_bounded_objective(objective_fn, bounds: np.ndarray):
    """
    Wrap objective so any x proposed by an unbounded optimizer (e.g. Nelder-Mead)
    is projected back into bounds before evaluation.
    """
    bounds = np.asarray(bounds, float)
    def obj(x):
        x = project_to_bounds(np.asarray(x, float), bounds)
        return objective_fn(x)
    return obj



# ============================================================
# 0) Config
# ============================================================

@dataclass
class MSMConfig:
    n_sim: int = 100_000
    age_start: int = 25
    age_end: int = 60
    seed: int = 123

    # Stage A: global Sobol screening
    sobol_draws: int = 250_000      # like Appendix D
    keep_best: int = 1_000          # keep best K points

    # Stage B: local optimization (derivative-free)
    local_methods: tuple = ("Powell", "Nelder-Mead")
    maxiter_local: int = 1_000

    # # Optional: keep  old interface fields (won't be used in new flow)
    # method: str = "L-BFGS-B"
    # maxiter: int = 500



# ============================================================
# 1) MSM building blocks: weights, psi, F, objective
# ============================================================

def make_diagonal_weights(n_mom: int, groups: dict) -> np.ndarray:
    """
    Build diagonal weights w (length n_mom) from groups.

    groups: dict like {
        "emp_cdf": (idx_array, group_weight),
        ...
    }

    Within each group, weight is split equally across its moments.
    """
    w = np.zeros(n_mom, dtype=float)
    for name, (idxs, group_weight) in groups.items():
        idxs = np.asarray(idxs, dtype=int)

        #skip if no target
        if idxs.size == 0:
            continue
            
        w[idxs] = float(group_weight) / idxs.size

    if np.any(w < 0) or np.isclose(w.sum(), 0.0):
        raise ValueError("Weights invalid. Check your group indices / weights.")
    return w


def compute_psi(m: np.ndarray, moment_sets: dict, floor: float = 1e-12) -> np.ndarray: #ALEX MIGHT NEED ADJUSTING
    """
    psi_n is the 10th percentile of |m_n| within each moment set.
    moment_sets: dict mapping set_name -> idx array
    """
    #convert moments to numpy
    m = np.asarray(m, float)

    # initialize psi
    psi = np.zeros_like(m)

    # abs value of moments
    abs_m = np.abs(m)

    for _, idxs in moment_sets.items():
        idxs = np.asarray(idxs, dtype=int)

        #skip if empyty sets
        if idxs.size == 0:
            continue

        # get the 10% moment ofr the set
        p10 = np.percentile(abs_m[idxs], 10)
        psi[idxs] = max(float(p10), floor)
        
    # handle moments not in any set
    psi[psi == 0] = floor  # any moment not covered by a set gets a tiny floor
    return psi


def deviation_F(d: np.ndarray, m: np.ndarray, psi: np.ndarray) -> np.ndarray: 
    """
    This is our Moment Deviation function:
    F_n(theta) = (d_n - m_n) / (0.5(|d_n| + |m_n|) + psi_n)
    """
    d = np.asarray(d, float)
    m = np.asarray(m, float)
    psi = np.asarray(psi, float)
    denom = 0.5 * (np.abs(d) + np.abs(m)) + psi
    return (d - m) / denom


# ============================================================
# 2) YOUR MODEL HOOKS (replace these two)
# ============================================================
def simulate_histories(theta: np.ndarray, cfg: MSMConfig, rng: np.random.Generator):
    """
    Vectorized Guvenen-style simulation for ages cfg.age_start..cfg.age_end.
    Returns dict with y_real (levels) and emp indicator (1 employed, 0 nonemp).
    """
    # ---- unpack theta (example ordering; adjust to your taste) ----
    (g_a, g_at, g_at2,
     sigma_alpha, sigma_beta, corr_alphaBeta,
     rho, sigma_z0,
     pz, mu_eta_1, sigma_eta_1, sigma_eta_2,
     peps, mu_eps_1, sigma_eps_1, sigma_eps_2,
     nu_a, nu_b, nu_c, nu_d,
     lmbda) = theta

    # ---- ages ----
    ages = np.arange(cfg.age_start, cfg.age_end + 1)
    T = ages.size
    t = (ages - 24) / 10.0  # Guvenen's scaling

    # ---- draw (alpha, beta) with correlation ----
    cov_alphaBeta = corr_alphaBeta * (sigma_alpha * sigma_beta)
    cov = np.array([[sigma_alpha**2, cov_alphaBeta],
                    [cov_alphaBeta, sigma_beta**2]], dtype=float)

    # n people (cfg.n_sim)
    n = cfg.n_sim

    ab = rng.multivariate_normal(mean=np.array([0.0, 0.0]), cov=cov, size=n)
    alpha = ab[:, 0:1]  # (n,1)
    beta  = ab[:, 1:2]  # (n,1)

    # ---- deterministic lifecycle g(t) ----
    gt = g_a + g_at * t + g_at2 * (t**2)               # (T,)
    gt = gt[None, :]                                   # (1,T)

    # ---- persistent shock z: AR(1) with mixture innovations ----
    # innovations eta ~ mixture of N(mu1,s1) and N(mu2,s2), mean zero overall
    mu_eta_2 = -mu_eta_1 * pz / (1.0 - pz)

    mix_u = rng.uniform(size=(n, T))
    eta = np.empty((n, T))
    # component 1
    idx1 = mix_u < pz
    eta[idx1] = rng.normal(loc=mu_eta_1, scale=sigma_eta_1, size=idx1.sum())
    # component 2
    eta[~idx1] = rng.normal(loc=mu_eta_2, scale=sigma_eta_2, size=(~idx1).sum())

    z = np.empty((n, T))
    z[:, 0] = rng.normal(loc=0.0, scale=sigma_z0, size=n)
    for j in range(1, T):
        z[:, j] = rho * z[:, j-1] + eta[:, j]

    # ---- transitory eps: mixture ----
    mu_eps_2 = -mu_eps_1 * peps / (1.0 - peps)

    mix_e = rng.uniform(size=(n, T))
    eps = np.empty((n, T))
    j1 = mix_e < peps
    eps[j1] = rng.normal(loc=mu_eps_1, scale=sigma_eps_1, size=j1.sum())
    eps[~j1] = rng.normal(loc=mu_eps_2, scale=sigma_eps_2, size=(~j1).sum())

    # ---- nonemployment shock nu ----
    # pnu = logistic(xi), xi depends on t, z, interaction
    xi = nu_a + nu_b * t[None, :] + nu_c * z + nu_d * (z * t[None, :])
    pnu = 1.0 / (1.0 + np.exp(-xi))

    nu_draw = rng.uniform(size=(n, T))
    # if nu_draw < pnu => draw nu ~ min(1, Exp(scale=1/lmbda)), else nu=0
    # scipy's expon(1/lmbda) is Exp(rate=lmbda) = Exp(scale=1/lmbda).
    # numpy exponential uses scale parameter.
    nu = np.zeros((n, T))
    take = nu_draw < pnu
    nu[take] = np.minimum(1.0, rng.exponential(scale=1.0/lmbda, size=take.sum()))

    # ---- earnings levels ----
    # Your current code: l2 = (1 - nu) * exp(g(t) + alpha + beta*t + z + eps)
    logy = gt + alpha + beta * t[None, :] + z + eps
    y = (1.0 - nu) * np.exp(logy)

    # employment indicator (you can define it as (nu==0) or (1-nu) if nu is intensity)
    emp = (y > 0.0).astype(float)  # or y > 1e-8
    
    return {"ages": ages, "t": t, "y_real": y, "emp": emp, "nu": nu, "z": z, "eps": eps}



def compute_model_moments(sim_data) -> np.ndarray:
    """
    Replace with your exact selection + moment construction.

    Must return d(theta) as a 1D array aligned with empirical moments m.
    """
    # -----------------------------
    # INSERT TOOL BOX HERE
    # -----------------------------
    


# ============================================================
# 3) MSM objective
# ============================================================

def msm_objective(theta: np.ndarray,
                  m: np.ndarray,
                  w_diag: np.ndarray,
                  psi: np.ndarray,
                  cfg: MSMConfig) -> float:
    """
    Objective: F(theta)' W F(theta), W diagonal with w_diag.
    Uses CRN via fixed seed inside objective for smoother optimization.
    """
    rng = np.random.default_rng(cfg.seed)
    sim_data = simulate_histories(theta, cfg, rng)
    d = compute_model_moments(sim_data)

    if d.shape != m.shape:
        raise ValueError(f"Model moments shape {d.shape} != empirical moments shape {m.shape}")

    F = deviation_F(d, m, psi)
    return float(np.sum(w_diag * (F ** 2)))


# ============================================================
# 4) Sobol multi-start
# ============================================================

def sobol_screen_and_refine(objective_fn,
                            bounds,
                            sobol_draws: int,
                            keep_best: int,
                            local_methods=("Powell", "Nelder-Mead"),
                            maxiter_local: int = 1000,
                            seed: int = 999):
    """
    Appendix-D style:
      1) Global stage: evaluate objective on many Sobol points
      2) Local stage: derivative-free local search from best points

    Returns
    -------
    results : list of scipy.optimize.OptimizeResult
        Local optimization results from the kept starting points, sorted by objective value.

    starts : np.ndarray
        All Sobol starting points of shape (sobol_draws, K).

    best_starts : np.ndarray
        The subset of starting points kept for local refinement of shape (keep_best, K).

    best_idx : np.ndarray
        Indices of best_starts in the starts array.
    """
    bounds = np.asarray(bounds, dtype=float)
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    K = bounds.shape[0]

    if bounds.ndim != 2 or bounds.shape[1] != 2:
        raise ValueError("bounds must be an array-like of shape (K, 2).")
    if np.any(~np.isfinite(lows)) or np.any(~np.isfinite(highs)):
        raise ValueError("Sobol screening requires finite bounds for all parameters.")
    if np.any(highs <= lows):
        raise ValueError("Each bound must satisfy upper > lower.")
    if sobol_draws <= 0:
        raise ValueError("sobol_draws must be positive.")
    if keep_best <= 0:
        raise ValueError("keep_best must be positive.")

    # Sobol sampler: random_base2 requires power of 2; use next power then slice
    sampler = qmc.Sobol(d=K, scramble=True, seed=seed)
    m_pow2 = int(np.ceil(np.log2(sobol_draws)))
    u = sampler.random_base2(m=m_pow2)[:sobol_draws]
    starts = qmc.scale(u, lows, highs)

    # Stage A: screen (evaluate objective once at each Sobol draw)
    vals = np.empty(sobol_draws, dtype=float)
    for i in range(sobol_draws):
        val = objective_fn(starts[i])
        vals[i] = val if np.isfinite(val) else np.inf


    # keep best keep_best
    keep_best = int(min(keep_best, sobol_draws))
    best_idx = np.argsort(vals)[:keep_best]
    best_starts = starts[best_idx]
    best_vals = vals[best_idx]

    # Stage B: local refine from best starts
    results = []
    bounded_obj = make_bounded_objective(objective_fn, bounds)

    for i, x0 in enumerate(best_starts):
        x_curr = x0.copy()
        best_res = None

        for method in local_methods:
            if method == "Powell":
                # Try Powell with bounds (supported in many SciPy versions).
                # If your SciPy doesn't support it, fallback to bounded objective.
                try:
                    res = minimize(
                        objective_fn,
                        x0=x_curr,
                        method="Powell",
                        bounds=[tuple(b) for b in bounds],
                        options={"maxiter": maxiter_local, "disp": False},
                    )
                except TypeError:
                    res = minimize(
                        bounded_obj,
                        x0=x_curr,
                        method="Powell",
                        options={"maxiter": maxiter_local, "disp": False},
                    )
                    res.x = project_to_bounds(res.x, bounds)
                    res.fun = objective_fn(res.x)

            elif method == "Nelder-Mead":
                res = minimize(
                    bounded_obj,
                    x0=x_curr,
                    method="Nelder-Mead",
                    options={"maxiter": maxiter_local, "disp": False},
                )
                # force final answer inside bounds + recompute true objective
                res.x = project_to_bounds(res.x, bounds)
                res.fun = objective_fn(res.x)

            else:
                raise ValueError(f"Unknown local method: {method}")

            # keep best result so far
            if best_res is None or (np.isfinite(res.fun) and res.fun < best_res.fun):
                best_res = res

            # warm-start next method from current best
            x_curr = best_res.x

        results.append(best_res)

    results.sort(key=lambda r: r.fun if np.isfinite(r.fun) else np.inf)
    return results, starts, best_starts, best_idx



# ============================================================
# 5) Fit wrapper
# ============================================================

def fit_msm_appendixD_style(m: np.ndarray,
                            bounds,
                            weights_groups: dict,
                            moment_sets: dict,
                            cfg: MSMConfig,
                            sobol_seed: int = 999):
    """
    Appendix D style estimation:
      - build W and psi
      - define MSM objective with CRN
      - Sobol screen many points
      - local derivative-free refinement from best points
    """
    m = np.asarray(m, float)
    n_mom = m.size

    w_diag = make_diagonal_weights(n_mom, weights_groups)
    psi = compute_psi(m, moment_sets)

    def obj(th):
        return msm_objective(th, m, w_diag, psi, cfg)

    results, all_starts, best_starts, best_idx = sobol_screen_and_refine(
        objective_fn=obj,
        bounds=bounds,
        sobol_draws=cfg.sobol_draws,
        keep_best=cfg.keep_best,
        local_methods=cfg.local_methods,
        maxiter_local=cfg.maxiter_local,
        seed=sobol_seed
    )

    best = results[0]
    return best, results, all_starts, best_starts, best_idx



In [ ]:
# For debugging
cfg = MSMConfig(n_sim=5000, sobol_draws=2048, keep_best=50, maxiter_local=300)
